In [4]:
import getpass
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()

 ········


In [6]:
os.environ["DEEPSEEK_API_KEY"] = getpass.getpass()

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-flash")

 ········


In [7]:
from langchain_core.messages import HumanMessage, SystemMessage
# SystemMessage —— 系统消息,用来给模型设定"角色/指令"
# HumanMessage —— 人类消息,代表用户输入

messages = [
    SystemMessage(content="Translate the following from English into chinese"),
    HumanMessage(content="You can only access an API key when you first create it. If you lost one, you will need to create a new one"),
]

model.invoke(messages)

AIMessage(content='只有在首次创建 API 密钥时才能查看它。如果丢失了，就需要创建一个新的。', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer user. Need translate English into Chinese. Need likely provide translation. We need parse: "You can only access an API key when you first create it. If you lost one, you will need to create a new one" Translate to Chinese. Need maybe "你只能在首次创建 API 密钥时访问它。如果丢失了，就需要创建一个新的。" Ensure API key translation: API 密钥. "访问" or "查看"? "You can only access an API key when you first create it" Means only when first create can you access/view it. Chinese: "只有在首次创建 API 密钥时，你才能获取/查看该密钥。如果已丢失，则需要创建一个新的。" Since "access" maybe "获取/访问". "access an API key" could be "查看 API 密钥". In API contexts, "You can only access an API key when you first create it" means you can only see/copy it at creation. Translation: "只有在首次创建 API 密钥时才能查看/获取它。如果密钥丢失，则需要创建一个新的。" Need maybe "API 密钥只能在首次创建时查看。如果丢失了，你需要创建一个新的。" That\'s natural. User asked translate from English into chinese. We can output just

In [8]:
# 输出解析器
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [9]:
## 使用解析器的一种方法是：单独使用
result = model.invoke(messages)

parser.invoke(result)

'你只能在首次创建 API 密钥时访问它。如果你丢失了密钥，就需要创建一个新的。'

In [10]:
## 更常见的使用方法是：将模型与解析器"链式"使用
## 我们可以使用 | 运算符轻松创建链。| 运算符在 LangChain 中用于将两个元素组合在一起。
chain = model | parser

chain.invoke(messages)

'你只能在首次创建 API 密钥时获取它。如果丢失了，就需要创建一个新的。'

In [11]:
# 提示词模板
## PromptTemplates 是 LangChain 中的一个概念
## 它们接收原始用户输入并返回准备传递给语言模型的数据（提示）
from langchain_core.prompts import ChatPromptTemplate

In [12]:
## 创建一个系统信息模板
system_template = "Translate the following into {language}"

In [13]:
## 接下来，我们可以创建 PromptTemplate
## 这将是 system_template 和一个更简单的模板的组合，用于放置要翻译的文本
prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

In [15]:
## 此提示词模板的输入是一个字典
result = prompt_template.invoke({"language": "chinese", "text": "Hello!How is it going!"})

result

ChatPromptValue(messages=[SystemMessage(content='Translate the following into chinese', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello!How is it going!', additional_kwargs={}, response_metadata={})])

In [16]:
result.to_messages()

[SystemMessage(content='Translate the following into chinese', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello!How is it going!', additional_kwargs={}, response_metadata={})]

In [17]:
# 使用 LCEL 连接组件
## 我们现在可以使用管道 (|) 操作符将其与上面的模型和输出解析器结合起来
chain = prompt_template | model | parser

In [18]:
chain.invoke({"language": "chinese", "text": "Hello!How is it going!"})

'你好！最近怎么样？'